# AI Skin Intelligence — Phase 7: Model Training

**EfficientNetB0 Transfer Learning on HAM10000 Skin Lesion Dataset**

---

## Before you start

1. **Enable GPU runtime**: Runtime → Change runtime type → T4 GPU → Save
2. **Set your Kaggle API token**: Paste your `KGAT_...` token in Step 1 below
3. **Run all cells in order** (Runtime → Run all)

Expected total time: ~45–90 minutes on a free Colab T4 GPU

---

### Dataset: HAM10000
- 10,015 dermatoscopic images across 7 skin lesion classes
- License: CC BY-NC 4.0 (non-commercial/educational use)
- Source: Kaggle — `kmader/skin-lesion-analysis-toward-melanoma-detection`

### Model: EfficientNetB0
- Pretrained on ImageNet (~5.3M parameters)
- Phase 1: Train only the classifier head (backbone frozen)
- Phase 2: Fine-tune the top backbone layers with a small LR

> ⚠️ **Disclaimer**: This is for educational purposes only. Not a medical diagnostic tool.

## Step 0 — Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'✅ GPU available: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU found! Go to Runtime → Change runtime type → T4 GPU')
    print('    Training on CPU will take many hours.')

## Step 1 — Kaggle Authentication and Dataset Download

**How to get your Kaggle API token (new `KGAT_...` format):**
1. Go to https://www.kaggle.com → Your Profile → Settings → API
2. Click **Create New Token** — your browser will show a token string starting with `KGAT_`
3. Copy that entire token string
4. Paste it below where it says `PASTE_YOUR_KAGGLE_TOKEN_HERE`

> ⚠️ **Security**: Never share your token or commit it to GitHub. It is only held in Colab's runtime memory and is erased when the session ends.

In [ ]:
import os

# ── Paste your Kaggle API token below (starts with KGAT_...) ──────────────────
# Get it from: kaggle.com → Profile → Settings → API → Create New Token
# This value is ONLY set in Colab memory — it is never saved to disk or Git.
os.environ['KAGGLE_API_TOKEN'] = 'PASTE_YOUR_KAGGLE_TOKEN_HERE'

# Verify the token was set (shows only the first 10 characters for safety)
token = os.environ.get('KAGGLE_API_TOKEN', '')
if token and token != 'PASTE_YOUR_KAGGLE_TOKEN_HERE':
    print(f'✅ Kaggle token set: {token[:10]}... ({len(token)} chars)')
else:
    raise ValueError('❌ Please paste your actual Kaggle token above before running this cell.')

In [ ]:
# Install / upgrade the Kaggle CLI to ensure KAGGLE_API_TOKEN support
!pip install -q -U kaggle

# ── Authentication test ────────────────────────────────────────────────────────
# This searches for HAM10000 on Kaggle. If authentication fails you will see
# a 401 error here — fix your token before proceeding to the download step.
print('Testing Kaggle authentication...')
!kaggle datasets list -s HAM10000

# ── Download ───────────────────────────────────────────────────────────────────
# Creates the destination directory and downloads + unzips the full dataset.
# Dataset: kmader/skin-lesion-analysis-toward-melanoma-detection (~2.4 GB)
!mkdir -p /content/ml/dataset/HAM10000

print('\nDownloading HAM10000 dataset (~2.4 GB)...')
print('This may take 5–15 minutes depending on Colab server speed.')

!kaggle datasets download \
    -d kmader/skin-lesion-analysis-toward-melanoma-detection \
    -p /content/ml/dataset/HAM10000 \
    --unzip

print('\n✅ Download complete! Contents:')
!ls /content/ml/dataset/HAM10000/

In [ ]:
# Consolidate images into a single folder
# The dataset has two parts: HAM10000_images_part_1 and HAM10000_images_part_2
import shutil, glob, os

images_dir = '/content/ml/dataset/HAM10000/images'
os.makedirs(images_dir, exist_ok=True)

# Copy all images from part 1 and part 2 into the unified images/ folder
for part in ['HAM10000_images_part_1', 'HAM10000_images_part_2']:
    part_path = f'/content/ml/dataset/HAM10000/{part}'
    if os.path.isdir(part_path):
        for img_file in glob.glob(f'{part_path}/*.jpg'):
            shutil.copy(img_file, images_dir)
        print(f'Copied images from {part}')

n_images = len(glob.glob(f'{images_dir}/*.jpg'))
print(f'\n✅ Total images in unified folder: {n_images}')
print(f'   Expected: 10,015')

## Step 2 — Clone Project and Set Up Source Code

In [ ]:
import sys

# Set up the ml/ source code inline (no git clone needed)
# We write the config to point at the Colab paths
ML_ROOT = '/content/ml'
sys.path.insert(0, ML_ROOT)

# Verify the source files exist
for fname in ['src/config.py', 'src/dataset.py', 'src/model.py', 'src/train.py', 'src/evaluate.py', 'src/predict.py']:
    path = f'{ML_ROOT}/{fname}'
    status = '✅' if os.path.exists(path) else '❌ MISSING'
    print(f'{status}  {fname}')

print('\nNote: If any files are missing, upload the ml/src/ folder to /content/ml/src/')

In [ ]:
# Install any missing libraries
!pip install scikit-learn matplotlib seaborn tqdm -q
print('✅ Libraries ready')

## Step 3 — Explore the Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Load metadata
metadata_csv = '/content/ml/dataset/HAM10000/HAM10000_metadata.csv'
df = pd.read_csv(metadata_csv)
print(f'Total records: {len(df)}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nClass distribution:')
print(df['dx'].value_counts())

In [ ]:
# Visualise class distribution
class_labels = {
    'akiec': 'Actinic Keratoses', 'bcc': 'Basal Cell Carcinoma',
    'bkl': 'Benign Keratosis', 'df': 'Dermatofibroma',
    'mel': 'Melanoma', 'nv': 'Melanocytic Nevi', 'vasc': 'Vascular Lesions'
}
counts = df['dx'].value_counts()
labels = [class_labels.get(k, k) for k in counts.index]

plt.figure(figsize=(10, 4))
bars = plt.bar(labels, counts.values, color='steelblue', edgecolor='white')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Number of images')
plt.title('HAM10000 — Class Distribution (Note: heavily imbalanced)')
for bar, count in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             str(count), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=100)
plt.show()
print('Note: NV dominates with ~67% of images — class weighting is critical!')

In [ ]:
# Preview sample images from each class
import glob

fig, axes = plt.subplots(1, 7, figsize=(18, 3))
class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

for ax, cls in zip(axes, class_names):
    sample_id = df[df['dx'] == cls]['image_id'].iloc[0]
    img_path = f'/content/ml/dataset/HAM10000/images/{sample_id}.jpg'
    if os.path.exists(img_path):
        img = mpimg.imread(img_path)
        ax.imshow(img)
    ax.set_title(f'{cls}\n{class_labels[cls][:15]}', fontsize=8)
    ax.axis('off')

plt.suptitle('Sample image from each class', y=1.02)
plt.tight_layout()
plt.savefig('/content/sample_images.png', dpi=100, bbox_inches='tight')
plt.show()

## Step 4 — Load Dataset and Build DataLoaders

In [ ]:
from src.dataset import get_dataloaders

train_loader, val_loader, test_loader, class_weights = get_dataloaders(
    metadata_csv='/content/ml/dataset/HAM10000/HAM10000_metadata.csv',
    images_dir='/content/ml/dataset/HAM10000/images',
    batch_size=32,
    num_workers=2,
)

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')
print(f'\nClass weights: {class_weights.tolist()}')

# Verify a batch shape
images, labels = next(iter(train_loader))
print(f'\nBatch shape: images={images.shape}, labels={labels.shape}')

## Step 5 — Build and Inspect Model

In [ ]:
from src.model import build_model, freeze_backbone, get_device, count_parameters

device = get_device()
model  = build_model(num_classes=7)
model  = model.to(device)

params = count_parameters(model)
print(f'\nModel: EfficientNetB0 (7 classes)')
print(f'  Total parameters     : {params["total"]:,}')
print(f'  Trainable parameters : {params["trainable"]:,}')

# Verify forward pass
dummy = torch.zeros(1, 3, 224, 224, device=device)
out   = model(dummy)
print(f'  Output shape         : {out.shape}  (expected: [1, 7])')

## Step 6 — Phase 1: Feature Extraction Training

Backbone is **frozen** — only the 7-class head is trained.
This is fast (~3–5 minutes per epoch on T4).

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import time

freeze_backbone(model)

class_weights = class_weights.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-7, verbose=True)

PHASE1_EPOCHS = 5
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'phase': []}

from src.train import train_one_epoch, evaluate, save_checkpoint, plot_history

print(f'Phase 1 — Feature Extraction ({PHASE1_EPOCHS} epochs)')
print('=' * 50)

for epoch in range(1, PHASE1_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['phase'].append(1)

    print(f'\nEpoch {epoch:02d}/{PHASE1_EPOCHS} [{elapsed:.0f}s]  '
          f'Train: loss={train_loss:.4f} acc={train_acc*100:.1f}%  |  '
          f'Val: loss={val_loss:.4f} acc={val_acc*100:.1f}%')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        os.makedirs('/content/ml/models', exist_ok=True)
        save_checkpoint(model, '/content/ml/models/efficientnetb0_ham10000.pt',
                        metadata={'phase': 1, 'epoch': epoch, 'val_loss': val_loss, 'val_acc': val_acc})

print('\nPhase 1 complete!')

## Step 7 — Phase 2: Fine-tuning

Unfreeze the top 20% of backbone layers and train with a very small LR (1e-5).
This adapts the high-level features to skin images without destroying pretrained weights.

In [ ]:
from src.model import unfreeze_top_layers

unfreeze_top_layers(model, fraction=0.2)

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7, verbose=True)

PHASE2_EPOCHS = 10
no_improve = 0
EARLY_STOP = 5

print(f'Phase 2 — Fine-tuning ({PHASE2_EPOCHS} epochs max)')
print('=' * 50)

for epoch in range(1, PHASE2_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['phase'].append(2)

    print(f'\nEpoch {epoch:02d}/{PHASE2_EPOCHS} [{elapsed:.0f}s]  '
          f'Train: loss={train_loss:.4f} acc={train_acc*100:.1f}%  |  '
          f'Val: loss={val_loss:.4f} acc={val_acc*100:.1f}%')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        save_checkpoint(model, '/content/ml/models/efficientnetb0_ham10000.pt',
                        metadata={'phase': 2, 'epoch': epoch, 'val_loss': val_loss, 'val_acc': val_acc})
    else:
        no_improve += 1
        if no_improve >= EARLY_STOP:
            print(f'\nEarly stopping after epoch {epoch} (no val improvement for {EARLY_STOP} epochs)')
            break

print('\nPhase 2 complete!')

## Step 8 — Plot Training Curves

In [ ]:
import json

plot_history(history, '/content/ml/models/training_curves.png')

with open('/content/ml/models/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print('Training history saved.')

# Display the plot
from IPython.display import Image
Image('/content/ml/models/training_curves.png')

## Step 9 — Evaluate on Test Set

In [ ]:
# Override config paths to use Colab paths before importing evaluate
import src.config as cfg
cfg.MODEL_PATH   = '/content/ml/models/efficientnetb0_ham10000.pt'
cfg.MODELS_DIR   = '/content/ml/models'
cfg.IMAGES_DIR   = '/content/ml/dataset/HAM10000/images'
cfg.METADATA_CSV = '/content/ml/dataset/HAM10000/HAM10000_metadata.csv'

from src.evaluate import evaluate
evaluate()

In [ ]:
# Display confusion matrix
from IPython.display import Image
Image('/content/ml/models/confusion_matrix.png')

## Step 10 — Test the Prediction Function

This verifies `predict.py` works correctly before we download the model.

In [ ]:
import importlib
import src.predict as pred_module

# Override MODEL_PATH in predict module to Colab path
import src.config as cfg
cfg.MODEL_PATH = '/content/ml/models/efficientnetb0_ham10000.pt'
importlib.reload(pred_module)
from src.predict import predict_image, predict_top_k

# Use a sample image from the test set
import glob, random
sample_images = glob.glob('/content/ml/dataset/HAM10000/images/*.jpg')
test_image = random.choice(sample_images)
print(f'Testing with: {os.path.basename(test_image)}')

result = predict_image(test_image)
print(f'\n✅ Prediction:')
print(f'   Class      : {result["class"]}')
print(f'   Label      : {result["label"]}')
print(f'   Confidence : {result["confidence"]*100:.2f}%')

print(f'\nTop-3 predictions:')
for p in predict_top_k(test_image, k=3):
    print(f'  {p["class"]:<8}: {p["confidence"]*100:.1f}%  {p["label"]}')

## Step 11 — Download Model for Local Use

Download the trained model file to your computer, then place it in:
```
AI-Skin-Intelligence/ml/models/efficientnetb0_ham10000.pt
```

In [ ]:
from google.colab import files

model_path = '/content/ml/models/efficientnetb0_ham10000.pt'
model_size = os.path.getsize(model_path) / 1e6
print(f'Model file size: {model_size:.1f} MB')

files.download(model_path)
files.download('/content/ml/models/training_history.json')
files.download('/content/ml/models/training_curves.png')
files.download('/content/ml/models/confusion_matrix.png')
files.download('/content/ml/models/test_metrics.json')

print('\n✅ All files downloaded!')
print('Place efficientnetb0_ham10000.pt in: AI-Skin-Intelligence/ml/models/')

## ✅ Phase 7 Complete!

You now have:
- A trained EfficientNetB0 model for 7-class skin lesion classification
- Full test-set evaluation metrics (accuracy, F1, confusion matrix)
- A reusable `predict_image()` function

**Next: Phase 8 — Connect the model to FastAPI**

> ⚠️ **Reminder**: This model is for **educational purposes only**. It is not a substitute for professional dermatological diagnosis.